# PerSAM-F: One-Shot Personalized Plant Disease Lesion Segmentation

> **Concept:** **PerSAM-F** (*Personalize Segment Anything Model with Fine-Tuning*) adapts SAM to segment any plant disease pathology using only **a single reference image and its binary mask** (one-shot).
>
> While zero-shot foundation models (like SAM) lack domain knowledge of crop pathologies:
> 1. **Visual Prior Extraction:** Extracts a compact reference feature embedding $\mathbf{v}_{ref}$ from the 1-shot reference pair representing the exact visual appearance of the lesion.
> 2. **Automated Point Prompting:** Computes a pixel-wise cosine feature similarity map $S$ across target leaves to automatically place optimal positive and negative point prompts.
> 3. **Fast 10-Second Optimization:** Fine-tunes **learnable multi-scale mask aggregation weights** on the single reference image, resolving scale ambiguities between lesion spots and the full leaf blade.
>
> **Adaptable Multi-Class Pipeline:** Select **any crop and disease class** below (e.g. `Apple___Black_rot`, `Tomato___Early_blight`, `Corn_(maize)___Common_rust_`, `Apple___Cedar_apple_rust`, etc.). All directories, prompts, one-shot reference pairs, generated pseudo-masks, quality controls, and manifests will dynamically configure to your chosen class.


## 1. Class Selection and Global Configuration
Configure your target disease class, reference pair paths, model backbone, and batching parameters below.
- **Interactive Class Selection:** Pick any disease class from the dropdown or type a custom class name.
- **Per-Class Output Directories:** Results are cleanly segregated by class under `/content/drive/MyDrive/plant-disease/persam_f/<class_name>/`.
- **Google Colab Optimization:** Use `EXECUTION_MODE = "chunk"` to process slices of 200-250 images safely within free Colab T4 limits.


In [ ]:
# Important: Colab comes with CUDA-optimized PyTorch and Torchvision pre-installed.
# Do NOT include 'torchvision' or 'torch' here, as pip -U installs generic PyPI wheels
# that break CUDA binary bindings and trigger 'RuntimeError: operator torchvision::nms does not exist'.
%pip install -q -U "transformers>=4.40.0" datasets accelerate safetensors opencv-python-headless \
     pandas matplotlib seaborn tqdm pyyaml


In [ ]:
from pathlib import Path
import os, random, time, json, shutil, warnings, re
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image

# ==============================================================================
# 1. DATASET & TARGET CLASS SELECTION (ADAPTABLE)
# ==============================================================================
DATASET_ID = "geraldmc/plantvillage-full"

# Select any disease class from PlantVillage (or choose 'CUSTOM'):
# Form parameter for Google Colab / interactive environments:
TARGET_CLASS_LABEL = "Apple___Black_rot"  # @param ["Apple___Black_rot", "Apple___Apple_scab", "Apple___Cedar_apple_rust", "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot", "Corn_(maize)___Common_rust_", "Corn_(maize)___Northern_Leaf_Blight", "Tomato___Bacterial_spot", "Tomato___Early_blight", "Tomato___Late_blight", "Tomato___Leaf_Mold", "Tomato___Septoria_leaf_spot", "Tomato___Spider_mites Two-spotted_spider_mite", "Tomato___Target_Spot", "Tomato___Tomato_Yellow_Leaf_Curl_Virus", "Tomato___Tomato_mosaic_virus", "CUSTOM"] {allow-input: true}

# If TARGET_CLASS_LABEL is "CUSTOM", specify your custom label here (e.g. "Grape___Black_rot", "Potato___Early_blight"):
CUSTOM_CLASS_LABEL = ""

# Standard project mapping for known classes
CLASS_MAP = {
    "Apple___Apple_scab": "apple__apple_scab",
    "Apple___Black_rot": "apple__black_rot",
    "Apple___Cedar_apple_rust": "apple__cedar_apple_rust",
    "Apple___healthy": "apple__healthy",
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot": "corn__cercospora_leaf_spot_gray_leaf_spot",
    "Corn_(maize)___Common_rust_": "corn__common_rust",
    "Corn_(maize)___Northern_Leaf_Blight": "corn__northern_leaf_blight",
    "Corn_(maize)___healthy": "corn__healthy",
    "Tomato___Bacterial_spot": "tomato__bacterial_spot",
    "Tomato___Early_blight": "tomato__early_blight",
    "Tomato___Late_blight": "tomato__late_blight",
    "Tomato___Leaf_Mold": "tomato__leaf_mold",
    "Tomato___Septoria_leaf_spot": "tomato__septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite": "tomato__spider_mites_two_spotted_spider_mite",
    "Tomato___Target_Spot": "tomato__target_spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus": "tomato__tomato_yellow_leaf_curl_virus",
    "Tomato___Tomato_mosaic_virus": "tomato__tomato_mosaic_virus",
    "Tomato___healthy": "tomato__healthy",
}

def resolve_class_info(label: str, custom: str = "") -> dict:
    """Dynamically parses and standardizes class label, crop name, and formatted name."""
    active_label = custom.strip() if label == "CUSTOM" and custom.strip() else label.strip()
    
    # 1. Standard map check
    if active_label in CLASS_MAP:
        std_name = CLASS_MAP[active_label]
        crop = std_name.split("__")[0]
        condition = std_name.split("__")[1]
    else:
        # Dynamic normalization for any arbitrary class label: "Crop___Condition"
        parts = active_label.split("___")
        if len(parts) == 2:
            raw_crop, raw_cond = parts
            crop = re.sub(r"[^a-zA-Z0-9]+", "_", raw_crop.split("_")[0]).strip("_").lower()
            cond = re.sub(r"[^a-zA-Z0-9]+", "_", raw_cond).strip("_").lower()
            std_name = f"{crop}__{cond}"
            condition = cond
        else:
            clean = re.sub(r"[^a-zA-Z0-9]+", "_", active_label).strip("_").lower()
            std_name = clean
            crop = clean.split("_")[0] if "_" in clean else "plant"
            condition = clean
            
    return {
        "raw_label": active_label,
        "class_name": std_name,
        "crop": crop,
        "condition": condition
    }

class_info = resolve_class_info(TARGET_CLASS_LABEL, CUSTOM_CLASS_LABEL)
TARGET_CLASS_LABEL = class_info["raw_label"]
TARGET_CLASS_NAME = class_info["class_name"]
CROP_NAME = class_info["crop"]
CONDITION_NAME = class_info["condition"]

print(f"Target Disease Selected: '{TARGET_CLASS_LABEL}'")
print(f"Standardized Name:       '{TARGET_CLASS_NAME}'")
print(f"Crop:                    '{CROP_NAME}'")
print(f"Pathology / Condition:   '{CONDITION_NAME}'")

# SAM Model ID (Open weights from Hugging Face Transformers)
# Options: "facebook/sam-vit-base" (fast, lightweight), "facebook/sam-vit-huge" (maximum quality)
SAM_MODEL_ID = "facebook/sam-vit-base"

# ==============================================================================
# 2. ONE-SHOT REFERENCE PAIR CONFIGURATION (PER-CLASS PATHS)
# ==============================================================================
BASE_DRIVE_DIR = Path("/content/drive/MyDrive/plant-disease")

# Store reference images and masks in a class-specific folder:
REF_DIR = BASE_DRIVE_DIR / "reference" / TARGET_CLASS_NAME
CUSTOM_REF_IMAGE_PATH = REF_DIR / f"{TARGET_CLASS_NAME}_ref.jpg"
CUSTOM_REF_MASK_PATH = REF_DIR / f"{TARGET_CLASS_NAME}_mask.png"

# Backward compatibility: check if reference was previously saved in legacy flat folder:
LEGACY_REF_IMAGE = BASE_DRIVE_DIR / "reference" / f"{TARGET_CLASS_NAME}_ref.jpg"
LEGACY_REF_MASK = BASE_DRIVE_DIR / "reference" / f"{TARGET_CLASS_NAME}_mask.png"
if not CUSTOM_REF_IMAGE_PATH.exists() and LEGACY_REF_IMAGE.exists():
    CUSTOM_REF_IMAGE_PATH = LEGACY_REF_IMAGE
if not CUSTOM_REF_MASK_PATH.exists() and LEGACY_REF_MASK.exists():
    CUSTOM_REF_MASK_PATH = LEGACY_REF_MASK

# Index of sample image within the target class to use if bootstrapping reference pair:
BOOTSTRAP_SAMPLE_INDEX = 0

# ==============================================================================
# 3. PERSAM-F FINE-TUNING HYPERPARAMETERS
# ==============================================================================
FINE_TUNE_STEPS = 250        # Optimization steps for learnable weights (~5-10s on GPU)
FINE_TUNE_LR = 1e-3          # Learning rate for 3-scale aggregation weights
SEED = 42

# Post-processing & Quality Control Thresholds
SCORE_THRESHOLD = 0.50       # Mask binarization confidence threshold
MIN_COMPONENT_PIXELS = 20    # Noise reduction: remove pixel blobs < 20 px
MIN_LESION_FRAC = 0.0005     # Min lesion area fraction (0.05% of leaf)
MAX_LESION_FRAC = 0.45       # Max lesion area fraction (45% of leaf)

# ==============================================================================
# 4. BATCHING & STORAGE CONFIGURATION (Google Colab T4 Safe)
# ==============================================================================
# "chunk" : processes a specific slice of indices (recommended for Colab T4)
# "pilot" : processes 20 test images
# "all"   : processes all images of the selected class in one run
EXECUTION_MODE = "chunk"
CHUNK_START = 0              # Start index for current session
CHUNK_SIZE = 250             # Process 250 images per session (~15-20 min on T4)
CHECKPOINT_INTERVAL = 25     # Save CSV to Drive every 25 images

SAVE_ORIGINAL_IMAGES = True  # Save original image copy for inspection
SAVE_OVERLAYS = False        # Overlays generated on the fly during review

# Output Directories on Google Drive organized by class
OUT_DIR = BASE_DRIVE_DIR / "persam_f" / TARGET_CLASS_NAME
IMAGE_DIR = OUT_DIR / "images"
MASK_DIR = OUT_DIR / "masks"
OVERLAY_DIR = OUT_DIR / "overlays"
METADATA_PATH = OUT_DIR / "metadata.csv"
MANIFEST_PATH = OUT_DIR / f"manifest_{TARGET_CLASS_NAME}.csv"
REVIEW_PATH = OUT_DIR / "review.csv"

created_dirs = [REF_DIR, OUT_DIR, MASK_DIR]
if SAVE_ORIGINAL_IMAGES:
    created_dirs.append(IMAGE_DIR)
if SAVE_OVERLAYS:
    created_dirs.append(OVERLAY_DIR)

for d in created_dirs:
    d.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("\nAdaptable Configuration initialized successfully!")
print(f"Output folder:    {OUT_DIR}")
print(f"Reference folder: {REF_DIR}")


## 2. Mount Google Drive and Load SAM
Loads the official `SamModel` and `SamProcessor` from Hugging Face Transformers. The backbone is loaded in `eval()` mode and frozen.


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Google Drive mounted successfully.")
except Exception as e:
    print("Running in local/non-Colab environment:", e)

from transformers import SamModel, SamProcessor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Computing device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

print(f"Loading SAM model: {SAM_MODEL_ID}...")
processor = SamProcessor.from_pretrained(SAM_MODEL_ID)
model = SamModel.from_pretrained(SAM_MODEL_ID).to(device)

# Freeze entire SAM model
for param in model.parameters():
    param.requires_grad = False
model.eval()

print("SAM model loaded and parameters successfully frozen!")


## 3. Dataset Ingestion & Class Validation
Loads `geraldmc/plantvillage-full` and filters all images corresponding to the selected target disease class: **`{TARGET_CLASS_LABEL}`** (`{TARGET_CLASS_NAME}`).


In [ ]:
from datasets import load_dataset, DatasetDict
import difflib

def load_plantvillage_dataset():
    try:
        ds = load_dataset(DATASET_ID)
    except Exception as exc:
        raise RuntimeError(f"Failed to load {DATASET_ID}. Check connection.") from exc
    if isinstance(ds, DatasetDict):
        split = "train" if "train" in ds else list(ds.keys())[0]
        return ds[split]
    return ds

dataset = load_plantvillage_dataset()
print("Dataset size:", len(dataset))
print("Available columns:", dataset.column_names)

label_col = "class_label" if "class_label" in dataset.column_names else None
if label_col is None:
    raise KeyError("Expected 'class_label' column not found in dataset.")

# Efficiently extract labels column without decoding images
all_labels = dataset[label_col]
label_counts = Counter(all_labels)

print(f"\nTotal unique classes in dataset: {len(label_counts)}")

# Validate target class label
if TARGET_CLASS_LABEL not in label_counts:
    matches = difflib.get_close_matches(TARGET_CLASS_LABEL, label_counts.keys(), n=5, cutoff=0.3)
    error_msg = f"Target class '{TARGET_CLASS_LABEL}' not found in dataset!\n"
    if matches:
        error_msg += f"Did you mean one of these?\n" + "\n".join(f" - {m}" for m in matches)
    else:
        error_msg += "Available classes:\n" + "\n".join(f" - {c}" for c in sorted(label_counts.keys())[:15]) + "\n..."
    raise ValueError(error_msg)

# Filter indices for the selected target class
target_class_indices = [
    i for i, lbl in enumerate(all_labels)
    if lbl == TARGET_CLASS_LABEL
]

print(f"\n=======================================================")
print(f"Selected Class:   {TARGET_CLASS_LABEL} ({TARGET_CLASS_NAME})")
print(f"Total Images:     {len(target_class_indices)}")
print(f"Crop:             {CROP_NAME.capitalize()}")
print(f"Pathology:        {CONDITION_NAME.replace('_', ' ').capitalize()}")
print(f"=======================================================")

# Select indices based on execution mode
if EXECUTION_MODE == "pilot":
    selected_indices = target_class_indices[:20]
    print(f"[Mode PILOT] Selected {len(selected_indices)} test images.")
elif EXECUTION_MODE == "chunk":
    chunk_end = min(CHUNK_START + CHUNK_SIZE, len(target_class_indices))
    selected_indices = target_class_indices[CHUNK_START:chunk_end]
    print(f"[Mode CHUNK] Active slice: index {CHUNK_START} to {chunk_end} ({len(selected_indices)} images for this session).")
else:
    selected_indices = target_class_indices
    print(f"[Mode ALL] All {len(selected_indices)} images will be processed.")


## 4. One-Shot Reference Pair Setup (Image & Lesion Mask)
PerSAM-F requires **1 reference image** and its corresponding **binary ground-truth mask** of the target lesion.

* **User Reference:** Place your image at `CUSTOM_REF_IMAGE_PATH` and binary mask at `CUSTOM_REF_MASK_PATH` (under `/content/drive/MyDrive/plant-disease/reference/<class_name>/`).
* **Candidate Preview:** Below, sample images from the selected class are previewed to help you select a reference sample if desired (`BOOTSTRAP_SAMPLE_INDEX`).
* **Bootstrap Seed Fallback:** If custom files do not exist yet on Drive, an automatic color-anomaly heuristic extracts an initial seed mask so you can run the notebook immediately. You can download and refine the mask anytime!


In [ ]:
from io import BytesIO
import matplotlib.pyplot as plt
import cv2

def get_pil_image(value):
    if isinstance(value, Image.Image):
        return value.convert("RGB")
    if isinstance(value, dict) and "bytes" in value:
        return Image.open(BytesIO(value["bytes"])).convert("RGB")
    if isinstance(value, str):
        return Image.open(value).convert("RGB")
    raise TypeError(f"Unsupported image type: {type(value)}")

# Preview first few candidate images from the selected class
print(f"Previewing candidate reference images for '{TARGET_CLASS_LABEL}':")
n_preview = min(4, len(target_class_indices))
fig, p_axes = plt.subplots(1, n_preview, figsize=(4 * n_preview, 3.5))
p_axes = np.atleast_1d(p_axes)
for idx_i, ax in enumerate(p_axes):
    sample_idx = target_class_indices[idx_i]
    cand_img = get_pil_image(dataset[sample_idx]["image"])
    ax.imshow(cand_img)
    ax.set_title(f"Sample #{idx_i} (Dataset idx: {sample_idx})")
    ax.axis("off")
plt.suptitle(f"Candidate Reference Samples for {TARGET_CLASS_NAME}", fontsize=12)
plt.tight_layout()
plt.show()

# Check if user provided custom reference files
if CUSTOM_REF_IMAGE_PATH.exists() and CUSTOM_REF_MASK_PATH.exists():
    print(f"Loading user-provided reference pair from Google Drive:")
    print(f" - Image: {CUSTOM_REF_IMAGE_PATH}")
    print(f" - Mask:  {CUSTOM_REF_MASK_PATH}")
    ref_image = Image.open(CUSTOM_REF_IMAGE_PATH).convert("RGB")
    ref_mask_np = (np.asarray(Image.open(CUSTOM_REF_MASK_PATH).convert("L")) > 128).astype(bool)
else:
    print(f"Custom reference pair not found at {CUSTOM_REF_IMAGE_PATH}.")
    safe_sample_idx = min(BOOTSTRAP_SAMPLE_INDEX, len(target_class_indices) - 1)
    chosen_dataset_idx = target_class_indices[safe_sample_idx]
    print(f"--> Bootstrapping reference image from sample #{safe_sample_idx} (dataset index {chosen_dataset_idx})...")
    
    ref_image = get_pil_image(dataset[chosen_dataset_idx]["image"])
    ref_image.save(CUSTOM_REF_IMAGE_PATH)
    
    # Generalized disease lesion bootstrap heuristic:
    # Plant lesions deviate from the healthy green tissue (chlorosis/yellow, necrotic brown/black, rust orange, or pale spots).
    arr = np.asarray(ref_image).astype(float)
    h, w, _ = arr.shape
    
    # 1. Detect healthy green leaf blade tissue: G is dominant over R and B
    healthy_green = (arr[..., 1] > arr[..., 0] * 1.05) & (arr[..., 1] > arr[..., 2] * 1.05) & (arr.mean(axis=-1) > 35)
    
    # 2. Leaf silhouette (non-background pixels)
    non_bg = (arr.mean(axis=-1) > 25) & (arr.mean(axis=-1) < 235)
    
    # 3. Lesion candidates: non-green anomalies on the leaf blade
    lesion_cand = non_bg & (~healthy_green)
    
    # Noise cleanup on lesion candidates
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(lesion_cand.astype(np.uint8))
    
    valid_components = []
    total_px = h * w
    for c in range(1, n_labels):
        area = stats[c, cv2.CC_STAT_AREA]
        if 40 <= area <= total_px * 0.40:
            valid_components.append((c, area))
            
    if valid_components:
        valid_components.sort(key=lambda x: x[1], reverse=True)
        best_comp = valid_components[0][0]
        ref_mask_np = (labels == best_comp)
    else:
        # Fallback centered seed region
        y, x = np.ogrid[:h, :w]
        ref_mask_np = ((x - w/2)**2 / (w/8)**2 + (y - h/2)**2 / (h/8)**2) <= 1.0

    Image.fromarray((ref_mask_np * 255).astype(np.uint8)).save(CUSTOM_REF_MASK_PATH)
    print(f"--> Saved bootstrap reference pair to {REF_DIR}.")
    print("--> TIP: For highest segmentation quality, replace this mask with an exact hand-annotated mask anytime!")

# Visualize the Reference Pair
fig, ax = plt.subplots(1, 3, figsize=(13, 4.2))
ax[0].imshow(ref_image)
ax[0].set_title(f"Reference Image (I_ref)\n[{TARGET_CLASS_NAME}]")
ax[0].axis("off")

ax[1].imshow(ref_mask_np, cmap="gray")
ax[1].set_title("Reference Lesion Mask (M_ref)\n[Binary Ground-Truth]")
ax[1].axis("off")

ax[2].imshow(ref_image)
ax[2].imshow(ref_mask_np, cmap="Reds", alpha=0.55)
ax[2].set_title("Overlay (Reference Lesion Concept)")
ax[2].axis("off")

plt.tight_layout()
plt.show()


## 5. PerSAM Visual Prior & Cosine Feature Affinity
In PerSAM:
1. SAM's image encoder produces dense visual features $F_{ref} \in \mathbb{R}^{C \times H' \times W'}$ ($64 \times 64$).
2. The reference visual prior $\mathbf{v}_{ref} \in \mathbb{R}^C$ is computed by average pooling $F_{ref}$ inside the binary reference lesion mask:
   $$\mathbf{v}_{ref} = \frac{\sum_{i,j} M_{ref}[i,j] \cdot F_{ref}[:, i, j]}{\sum_{i,j} M_{ref}[i,j] + \epsilon}, \quad \hat{\mathbf{v}}_{ref} = \frac{\mathbf{v}_{ref}}{\|\mathbf{v}_{ref}\|_2}$$
3. For any target image, dense features $F_{target}$ are extracted. The pixel-wise cosine similarity map $S \in [-1, 1]$ is computed:
   $$S[i,j] = \hat{\mathbf{v}}_{ref} \cdot \frac{F_{target}[:, i, j]}{\|F_{target}[:, i, j]\|_2}$$
4. The location with maximum similarity $(x_{pos}, y_{pos}) = \arg\max S$ is selected as the positive foreground point prompt!


In [ ]:
def get_image_embeddings(img):
    """Passes an RGB PIL image through SAM's frozen image encoder."""
    inputs = processor(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        image_embeddings = model.get_image_embeddings(inputs["pixel_values"])
    return image_embeddings  # Shape: [1, 256, 64, 64]

def extract_reference_feature(ref_img, ref_mask_bool):
    """Computes normalized reference lesion feature vector v_ref."""
    ref_emb = get_image_embeddings(ref_img)  # [1, 256, 64, 64]
    
    # Downsample reference mask to 64x64 feature resolution
    mask_tensor = torch.tensor(ref_mask_bool, dtype=torch.float32, device=device).unsqueeze(0).unsqueeze(0)
    mask_down = F.interpolate(mask_tensor, size=(64, 64), mode="nearest")[0, 0]  # [64, 64]
    
    if mask_down.sum() == 0:
        # Fallback to center region if mask is tiny
        mask_down[30:34, 30:34] = 1.0

    # Area-weighted average feature vector
    feat_2d = ref_emb[0]  # [256, 64, 64]
    v_ref = (feat_2d * mask_down).sum(dim=(-2, -1)) / (mask_down.sum() + 1e-6)  # [256]
    v_ref = v_ref / (v_ref.norm(p=2) + 1e-6)
    return v_ref, ref_emb

def compute_similarity_map(target_emb, v_ref):
    """Computes dense cosine similarity map S in [64, 64]."""
    feat_2d = target_emb[0]  # [256, 64, 64]
    feat_norm = feat_2d / (feat_2d.norm(dim=0, keepdim=True) + 1e-6)  # [256, 64, 64]
    sim_map = torch.einsum("c,chw->hw", v_ref, feat_norm)  # [64, 64]
    return sim_map

def extract_prompts_from_sim_map(sim_map, img_width, img_height):
    """Extracts positive point (argmax) and negative point (argmin)."""
    h_feat, w_feat = sim_map.shape
    
    # Highest similarity = positive prompt point
    pos_idx = sim_map.flatten().argmax().item()
    pos_y_feat, pos_x_feat = divmod(pos_idx, w_feat)
    pos_x = float(pos_x_feat * img_width / w_feat)
    pos_y = float(pos_y_feat * img_height / h_feat)
    
    # Lowest similarity = negative prompt point
    neg_idx = sim_map.flatten().argmin().item()
    neg_y_feat, neg_x_feat = divmod(neg_idx, w_feat)
    neg_x = float(neg_x_feat * img_width / w_feat)
    neg_y = float(neg_y_feat * img_height / h_feat)
    
    input_points = [[[pos_x, pos_y], [neg_x, neg_y]]]
    input_labels = [[1, 0]]  # 1 = positive (lesion), 0 = background
    return input_points, input_labels, sim_map.detach().cpu().numpy()

# Extract reference visual feature
v_ref, ref_embeddings = extract_reference_feature(ref_image, ref_mask_np)
print("Reference feature vector shape:", v_ref.shape)

# Verify on reference image
ref_pts, ref_lbls, ref_sim = extract_prompts_from_sim_map(
    compute_similarity_map(ref_embeddings, v_ref), ref_image.width, ref_image.height
)
print("Extracted positive prompt point:", ref_pts[0][0])
print("Extracted negative prompt point:", ref_pts[0][1])


## 6. PerSAM-F: One-Shot 10-Second Fine-Tuning
SAM's mask decoder outputs **3 candidate masks** representing multi-scale granularities (whole, part, subpart).
In **PerSAM-F**, we freeze all of SAM and introduce **learnable mask aggregation weights** $\mathbf{w} = [w_1, w_2, w_3]$:
$$L_{aggregated} = \sum_{k=0}^2 \text{softmax}(\mathbf{w})_k \cdot L_k$$

Using the single reference pair $(I_{ref}, M_{ref})$, we optimize $\mathbf{w}$ using combined **Binary Cross-Entropy (BCE) and Dice Loss** for 250 steps.
This takes **less than 10 seconds** on GPU!


In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, eps=1e-5):
        super().__init__()
        self.eps = eps

    def forward(self, pred_probs, targets):
        intersection = (pred_probs * targets).sum()
        union = pred_probs.sum() + targets.sum()
        dice = (2.0 * intersection + self.eps) / (union + self.eps)
        return 1.0 - dice

class PerSAM_F_Head(nn.Module):
    """Learnable multi-scale mask weighting head for PerSAM-F."""
    def __init__(self):
        super().__init__()
        # Initialized equally (1/3 each)
        self.weights = nn.Parameter(torch.tensor([1.0, 1.0, 1.0], dtype=torch.float32))

    def forward(self, multi_scale_logits):
        # multi_scale_logits: [3, H, W]
        probs = F.softmax(self.weights, dim=0)
        combined = (probs.view(3, 1, 1) * multi_scale_logits).sum(dim=0)
        return combined, probs

# Step 1: Compute candidate masks for reference image using SAM Decoder
ref_inputs = processor(
    images=ref_image,
    input_points=ref_pts,
    input_labels=ref_lbls,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    ref_outputs = model(
        image_embeddings=ref_embeddings,
        input_points=ref_inputs["input_points"],
        input_labels=ref_inputs["input_labels"]
    )
    # Output pred_masks shape: [1, 1, 3, 256, 256]
    candidate_logits = ref_outputs.pred_masks[0, 0]  # [3, 256, 256]
    # Interpolate candidate logits to reference image resolution
    candidate_logits = F.interpolate(
        candidate_logits.unsqueeze(0),
        size=(ref_image.height, ref_image.width),
        mode="bilinear",
        align_corners=False
    )[0]  # [3, H, W]

# Ground-truth reference mask tensor
target_mask_tensor = torch.tensor(ref_mask_np, dtype=torch.float32, device=device)

# Step 2: Fine-tune PerSAM-F learnable head
persam_f_head = PerSAM_F_Head().to(device)
optimizer = torch.optim.Adam(persam_f_head.parameters(), lr=FINE_TUNE_LR)
bce_fn = nn.BCEWithLogitsLoss()
dice_fn = DiceLoss()

print(f"Starting PerSAM-F 1-shot fine-tuning for '{TARGET_CLASS_LABEL}' ({FINE_TUNE_STEPS} steps)...")
start_time = time.time()
loss_history = []

for step in range(1, FINE_TUNE_STEPS + 1):
    optimizer.zero_grad()
    pred_logit, current_weights = persam_f_head(candidate_logits)
    
    loss_bce = bce_fn(pred_logit, target_mask_tensor)
    loss_dice = dice_fn(torch.sigmoid(pred_logit), target_mask_tensor)
    loss = loss_bce + loss_dice
    
    loss.backward()
    optimizer.step()
    
    loss_history.append(loss.item())
    if step % 50 == 0 or step == FINE_TUNE_STEPS:
        print(f"Step {step:03d}/{FINE_TUNE_STEPS} | Total Loss: {loss.item():.4f} (BCE: {loss_bce.item():.4f}, Dice: {loss_dice.item():.4f}) | Weights: {current_weights.detach().cpu().numpy().round(3)}")

elapsed = time.time() - start_time
print(f"Fine-tuning complete in {elapsed:.2f} seconds!")

# Plot training loss
plt.figure(figsize=(7, 3))
plt.plot(loss_history, color="crimson", lw=2)
plt.title(f"PerSAM-F 1-Shot Fine-Tuning Loss Curve ({TARGET_CLASS_NAME})")
plt.xlabel("Step")
plt.ylabel("BCE + Dice Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 7. Qualitative Smoke Test: PerSAM Baseline vs. Fine-Tuned PerSAM-F
We evaluate the impact of PerSAM-F fine-tuning on unseen test images of the selected class.


In [ ]:
def predict_persam_f(target_img, use_finetuned=True):
    """Full PerSAM-F inference pipeline on a single image."""
    # 1. Embeddings & Cosine Similarity Map
    target_emb = get_image_embeddings(target_img)
    sim_map = compute_similarity_map(target_emb, v_ref)
    
    # 2. Extract prompts
    pts, lbls, _ = extract_prompts_from_sim_map(sim_map, target_img.width, target_img.height)
    
    # 3. SAM Decoder forward pass
    inputs = processor(images=target_img, input_points=pts, input_labels=lbls, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(
            image_embeddings=target_emb,
            input_points=inputs["input_points"],
            input_labels=inputs["input_labels"]
        )
        cand_logits = outputs.pred_masks[0, 0]  # [3, 256, 256]
        cand_logits = F.interpolate(
            cand_logits.unsqueeze(0),
            size=(target_img.height, target_img.width),
            mode="bilinear",
            align_corners=False
        )[0]
    
    if use_finetuned:
        pred_logit, _ = persam_f_head(cand_logits)
    else:
        # Standard PerSAM uses highest score mask
        scores = outputs.iou_scores[0, 0]
        best_idx = scores.argmax().item()
        pred_logit = cand_logits[best_idx]

    pred_mask = (torch.sigmoid(pred_logit) > SCORE_THRESHOLD).detach().cpu().numpy()
    return pred_mask, pts[0][0]

# Test on up to 3 sample images
test_indices = selected_indices[:min(3, len(selected_indices))]
fig, axes = plt.subplots(len(test_indices), 4, figsize=(16, 4 * len(test_indices)))
if len(test_indices) == 1:
    axes = np.array([axes])

for row_idx, idx in enumerate(test_indices):
    test_img = get_pil_image(dataset[idx]["image"])
    
    mask_baseline, pos_pt = predict_persam_f(test_img, use_finetuned=False)
    mask_persam_f, _ = predict_persam_f(test_img, use_finetuned=True)
    
    # 1. Original Image with prompt point
    axes[row_idx, 0].imshow(test_img)
    axes[row_idx, 0].plot(pos_pt[0], pos_pt[1], "go", markersize=8, markeredgecolor="white", label="Pos Prompt")
    axes[row_idx, 0].set_title(f"Target Image #{idx}\n(Argmax Similarity Point)")
    axes[row_idx, 0].axis("off")
    axes[row_idx, 0].legend(loc="lower right")
    
    # 2. Similarity Map
    sim = compute_similarity_map(get_image_embeddings(test_img), v_ref).detach().cpu().numpy()
    axes[row_idx, 1].imshow(sim, cmap="turbo")
    axes[row_idx, 1].set_title(f"Cosine Similarity Map\n[{TARGET_CLASS_NAME}]")
    axes[row_idx, 1].axis("off")
    
    # 3. Base PerSAM Mask
    axes[row_idx, 2].imshow(test_img)
    axes[row_idx, 2].imshow(mask_baseline, cmap="Blues", alpha=0.55)
    axes[row_idx, 2].set_title("Base PerSAM (Zero-Shot)")
    axes[row_idx, 2].axis("off")
    
    # 4. PerSAM-F Mask
    axes[row_idx, 3].imshow(test_img)
    axes[row_idx, 3].imshow(mask_persam_f, cmap="Reds", alpha=0.55)
    axes[row_idx, 3].set_title(f"PerSAM-F (Fine-Tuned)\n[{TARGET_CLASS_NAME}]")
    axes[row_idx, 3].axis("off")

plt.tight_layout()
plt.show()


## 8. Resumable Batch / Chunk Pseudo-Mask Generation
Applies the fine-tuned PerSAM-F model across the selected images for **`{TARGET_CLASS_LABEL}`**.
* **Resumable:** Reads `metadata.csv` on Google Drive to skip already processed images.
* **Checkpointed:** Saves progress every `CHECKPOINT_INTERVAL` (25 images).
* **Noise Suppression:** Removes isolated pixel components < `MIN_COMPONENT_PIXELS` px via OpenCV.


In [ ]:
import cv2
from tqdm.auto import tqdm

def clean_mask_components(mask_bool, min_pixels):
    """Removes small isolated noise components."""
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask_bool.astype(np.uint8), connectivity=8)
    clean_mask = np.zeros_like(mask_bool, dtype=bool)
    for c in range(1, n):
        if stats[c, cv2.CC_STAT_AREA] >= min_pixels:
            clean_mask[labels == c] = True
    return clean_mask

def evaluate_qc(mask, total_pixels):
    lesion_pixels = int(mask.sum())
    frac = lesion_pixels / total_pixels if total_pixels else 0.0
    
    reasons = []
    if lesion_pixels == 0:
        reasons.append("empty_mask")
    elif frac < MIN_LESION_FRAC:
        reasons.append("too_small")
    elif frac > MAX_LESION_FRAC:
        reasons.append("too_large")
        
    n_components = cv2.connectedComponents(mask.astype(np.uint8))[0] - 1 if lesion_pixels else 0
    return {
        "lesion_pixels": lesion_pixels,
        "lesion_area_frac": frac,
        "n_components": n_components,
        "qc_pass": len(reasons) == 0,
        "qc_reason": "|".join(reasons) if reasons else "ok"
    }

existing_ids = set()
if METADATA_PATH.exists():
    existing_ids = set(pd.read_csv(METADATA_PATH, usecols=["image_id"])["image_id"])
print(f"Existing processed images in Drive metadata: {len(existing_ids)}")

records = []
failures = []
started = time.time()
new_count = 0

for pos, index in enumerate(tqdm(selected_indices, desc=f"PerSAM-F [{TARGET_CLASS_NAME} - {EXECUTION_MODE}]")):
    row = dataset[index]
    image_id = f"{TARGET_CLASS_NAME}_{index:06d}"

    if image_id in existing_ids:
        continue

    try:
        image = get_pil_image(row["image"])
        raw_mask, pos_pt = predict_persam_f(image, use_finetuned=True)
        clean_mask = clean_mask_components(raw_mask, MIN_COMPONENT_PIXELS)
        qc = evaluate_qc(clean_mask, image.width * image.height)

        mask_path = MASK_DIR / f"{image_id}.png"
        image_path = IMAGE_DIR / f"{image_id}.jpg"
        overlay_path = OVERLAY_DIR / f"{image_id}.jpg"

        # Save compact binary mask (0 or 255)
        Image.fromarray((clean_mask * 255).astype(np.uint8)).save(mask_path)

        if SAVE_ORIGINAL_IMAGES:
            image.save(image_path, quality=95)

        if SAVE_OVERLAYS:
            overlay = np.asarray(image).copy()
            overlay[clean_mask] = (0.55 * overlay[clean_mask] + 0.45 * np.array([255, 0, 0])).astype(np.uint8)
            Image.fromarray(overlay).save(overlay_path, quality=92)

        rec = {
            "image_id": image_id,
            "dataset_index": index,
            "class_name": TARGET_CLASS_NAME,
            "crop": CROP_NAME,
            "condition": CONDITION_NAME,
            "image_path": str(image_path) if SAVE_ORIGINAL_IMAGES else "",
            "mask_path": str(mask_path),
            "overlay_path": str(overlay_path) if SAVE_OVERLAYS else "",
            "prompt_pos_x": pos_pt[0],
            "prompt_pos_y": pos_pt[1],
            "method": "PerSAM-F",
            "sam_backbone": SAM_MODEL_ID,
            **qc
        }
        records.append(rec)
        existing_ids.add(image_id)
        new_count += 1

        # Checkpoint to Drive every CHECKPOINT_INTERVAL
        if len(records) >= CHECKPOINT_INTERVAL:
            old = pd.read_csv(METADATA_PATH) if METADATA_PATH.exists() else pd.DataFrame()
            pd.concat([old, pd.DataFrame(records)], ignore_index=True).to_csv(METADATA_PATH, index=False)
            records.clear()

        # Periodically empty CUDA cache
        if device.type == "cuda" and pos % 50 == 0:
            torch.cuda.empty_cache()

    except Exception as exc:
        failures.append({"image_id": image_id, "error": repr(exc)})
        print(f"Error on {image_id}: {exc}")

# Final flush
if records:
    old = pd.read_csv(METADATA_PATH) if METADATA_PATH.exists() else pd.DataFrame()
    pd.concat([old, pd.DataFrame(records)], ignore_index=True).to_csv(METADATA_PATH, index=False)

if failures:
    fail_csv = OUT_DIR / "failures.csv"
    old_f = pd.read_csv(fail_csv) if fail_csv.exists() else pd.DataFrame()
    pd.concat([old_f, pd.DataFrame(failures)], ignore_index=True).to_csv(fail_csv, index=False)

print(f"Batch generation completed: {new_count} newly processed images.")
print("Session time (minutes):", round((time.time() - started) / 60, 2))


## 9. Quality Control Statistics
Summarizes pass/reject counts and metrics for the active target class.


In [ ]:
if METADATA_PATH.exists():
    meta = pd.read_csv(METADATA_PATH)
    print(f"Total records for '{TARGET_CLASS_LABEL}' ({TARGET_CLASS_NAME}):", len(meta))
    print("Passed QC:", int(meta["qc_pass"].sum()))
    print("Rejected QC:", int((~meta["qc_pass"]).sum()))
    print(f"Pass rate: {meta['qc_pass'].mean():.2%}")
    print("\nRejection Breakdown:")
    print(meta.loc[~meta.qc_pass, "qc_reason"].value_counts())
    print("\nLesion Area Fraction Summary:")
    print(meta["lesion_area_frac"].describe())
else:
    print(f"No metadata found yet at {METADATA_PATH}. Run the batch generation cell first.")


## 10. Visual Review Grid (Contact Sheet)
Generates and displays inspection grids showing predicted PerSAM-F masks overlaid on real target leaves.


In [ ]:
def generate_contact_sheet(df, title, filename, n=16):
    samples = df.sample(min(n, len(df)), random_state=SEED) if len(df) else df
    if samples.empty:
        print(f"No samples to display for {title}.")
        return
    rows = (len(samples) + 3) // 4
    fig, axes = plt.subplots(rows, 4, figsize=(16, 4 * rows))
    axes = np.atleast_1d(axes).ravel()
    
    for ax, (_, item) in zip(axes, samples.iterrows()):
        img_p = Path(item["image_path"]) if pd.notna(item.get("image_path")) and item.get("image_path") else None
        if img_p and img_p.exists():
            img = np.asarray(Image.open(img_p).convert("RGB"))
        else:
            # Fallback to dataset index
            img = np.asarray(get_pil_image(dataset[int(item["dataset_index"])]["image"]))
            
        mask = np.asarray(Image.open(item["mask_path"])) > 0
        ax.imshow(img)
        ax.imshow(mask, cmap="Reds", alpha=0.50)
        ax.set_title(f"{item['image_id']}\nQC: {item['qc_reason']} ({item['lesion_area_frac']:.1%})", fontsize=10)
        ax.axis("off")
        
    for ax in axes[len(samples):]:
        ax.axis("off")
        
    fig.suptitle(title, fontsize=15)
    fig.tight_layout()
    fig.savefig(OUT_DIR / filename, dpi=120)
    plt.show()

if METADATA_PATH.exists():
    meta = pd.read_csv(METADATA_PATH)
    generate_contact_sheet(
        meta[meta.qc_pass == True],
        f"PerSAM-F Passed Lesion Masks ({TARGET_CLASS_LABEL})",
        f"review_passed_{TARGET_CLASS_NAME}.jpg"
    )
    generate_contact_sheet(
        meta[meta.qc_pass == False],
        f"PerSAM-F Rejected Lesion Masks ({TARGET_CLASS_LABEL})",
        f"review_rejected_{TARGET_CLASS_NAME}.jpg"
    )


## 11. Export Manifests
Exports standard project manifests:
* `manifest_<class_name>.csv` : All generated records with QC flags.
* `manifest_training_<class_name>.csv` & `manifest_training.csv` : Clean, QC-passed records strictly for training downstream models.


In [ ]:
if METADATA_PATH.exists():
    meta = pd.read_csv(METADATA_PATH)
    export_cols = [
        "image_id", "dataset_index", "class_name", "crop",
        "image_path", "mask_path", "method", "sam_backbone",
        "lesion_pixels", "lesion_area_frac", "n_components", "qc_pass"
    ]
    valid_cols = [c for c in export_cols if c in meta.columns]
    
    # 1. Full manifest with QC flags
    meta[valid_cols].to_csv(MANIFEST_PATH, index=False)
    
    # 2. Training-ready manifests (QC passed only)
    training_manifest_class = OUT_DIR / f"manifest_training_{TARGET_CLASS_NAME}.csv"
    training_manifest_generic = OUT_DIR / "manifest_training.csv"
    
    meta[meta.qc_pass == True][valid_cols].to_csv(training_manifest_class, index=False)
    meta[meta.qc_pass == True][valid_cols].to_csv(training_manifest_generic, index=False)
    
    print("Full manifest exported to:            ", MANIFEST_PATH)
    print(f"Training manifest ({TARGET_CLASS_NAME}):", training_manifest_class)
    print("Generic training manifest:            ", training_manifest_generic)
    print(f"Ready for training with {meta.qc_pass.sum()} high-quality pseudo-masks for {TARGET_CLASS_NAME}!")
else:
    print("Metadata path does not exist.")
